# **Notebook 2: Data Preparation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `sampled_data.csv` — Created by Notebook 1
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] Internet access (to download the tokenizer)

**Files this notebook will CREATE:**
- [ ] `./tokenized_train/`, `./tokenized_valid/`, `./tokenized_test/` — HuggingFace datasets _(Required by NB6)_
- [ ] `df_test.csv` — Test set with readable text + labels _(Required by NB5 and NB7)_

---

## **Stage 2: Data Preparation**
### **Task 2.1: Clean and Validate Dataset**

#### **2.1.1 Remove Duplicate and Invalid Records [2 marks]**
**The Task:** Identify and remove duplicate records using normalised deduplication (lowercase + strip before comparison), plus any invalid, incomplete, or corrupted records.

**Hints & Tips:**
* Load `sampled_data.csv` from Notebook 1 first.
* Normalised dedup: create a temporary `_norm` column = `instruction.str.lower().str.strip()`, drop duplicates on it, then drop the helper column.
* Check for nulls in `instruction`, `intent`, and `category` — drop rows missing any of these targets.
* Print the number of rows removed so the cleaning is auditable.

**Why we are doing it:** To prevent degraded training from redundant or broken records, and to stop near-duplicates leaking across train/test splits later.
**How we are doing it:** Pandas normalised deduplication and null filtering.

**Learner Inference:** Exact-match dedup misses `"Where is my order?"` vs `"where is my order?"`. Normalising first catches these and keeps your splits clean.

In [ ]:
# YOUR CODE HERE
import pandas as pd
import os

# Load sampled_data.csv from Notebook 1
df = pd.read_csv("sampled_data.csv")
initial_len = len(df)
print(f"Loaded {initial_len} records from sampled_data.csv")

# Normalized deduplication: lowercase + strip whitespace
df['_norm'] = df['instruction'].astype(str).str.lower().str.strip()
df = df.drop_duplicates(subset=['_norm']).copy()
df = df.drop(columns=['_norm'])

# Drop nulls
df = df.dropna(subset=['instruction', 'intent', 'category']).copy()
print(f"Records after normalized deduplication and null validation: {len(df)} (Removed: {initial_len - len(df)})")

#### **2.1.2 Standardise Dataset Formatting [2 marks]**
**The Task:** Clean the strings inside `instruction`, `intent`, and `category` by stripping whitespace and enforcing uniform casing (intents = lower, categories = upper).

**Hints & Tips:**
* Use Pandas `.str.strip()`, `.str.lower()`, `.str.upper()`.
* Always strip whitespace BEFORE lowering — trailing spaces survive `.lower()`.
* Standardised casing prevents the model treating "Refund" and "refund" as different classes.

**Why we are doing it:** To ensure raw strings are perfectly standardised before injecting them into prompt templates.
**How we are doing it:** Pandas string manipulation.

**Learner Inference:** Standardising capitalisation prevents the model from treating the same intent written two ways as two separate classes.

In [ ]:
# YOUR CODE HERE
df['instruction'] = df['instruction'].astype(str).str.strip()
df['intent'] = df['intent'].astype(str).str.strip().str.lower()
df['category'] = df['category'].astype(str).str.strip().str.upper()

print("Standardised Casing Verification:")
print("Sample Intents (lowercase):", df['intent'].unique()[:5])
print("Sample Categories (uppercase):", df['category'].unique()[:5])
print(f"Cleaned dataset schema: {list(df.columns)}")

### **Task 2.2: Create Instruction-Tuning Dataset**

#### **2.2.1 Transform Records into Instruction Format [2 marks]**
**The Task:** Wrap the raw instructions and targets into a standardized `ChatML` format, forcing the assistant's response to be a strict JSON string containing the `intent` and `category`.

**Hints & Tips:**
* Use the Qwen tokenizer's `apply_chat_template(tokenize=False)`.
* The target JSON is `{"intent": "...", "category": "..."}` — built via `json.dumps()`.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — fits T4, strong instruction following.
* **TinyLlama-1.1B-Chat** — lighter but weaker at structured output.
* **Llama-3-8B-Instruct** — best quality, needs >16GB VRAM even in 4-bit.
* Whatever you pick, set `MODEL_ID` here and in ALL later notebooks.

**Learner Inference:** Formatting the target as JSON establishes the exact structural rules the LoRA model learns to mimic in Stage 4.

In [ ]:
# YOUR CODE HERE
import json
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def create_chatml_record(row):
    target_json = json.dumps({
        "intent": row["intent"],
        "category": row["category"]
    }, ensure_ascii=False)
    
    messages = [
        {"role": "system", "content": "You are an expert customer support intent classification system. Extract the intent and category from the user query and return strictly valid JSON."},
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": target_json}
    ]
    
    formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return formatted_text, target_json

formatted_records = [create_chatml_record(row) for _, row in df.iterrows()]
df['chatml_text'] = [r[0] for r in formatted_records]
df['target_json'] = [r[1] for r in formatted_records]

print("Sample Formatted ChatML Record:")
print(df['chatml_text'].iloc[0])

#### **2.2.2 Validate Instruction Templates [2 marks]**
**The Task:** Extract the assistant's portion of the generated ChatML string and parse it with `json.loads()` to verify structural integrity.

**Hints & Tips:**
* Split the string by `<|im_start|>assistant` and parse what follows.
* If this throws `JSONDecodeError`, your formatting in 2.2.1 is broken — fix before training.
* Validate several samples, not just the first.

**Learner Inference:** A JSONDecodeError here means the model would learn broken syntax. Catching it now saves a failed training run.

In [ ]:
# YOUR CODE HERE
valid_count = 0
for idx, row in df.iterrows():
    chatml = row['chatml_text']
    if "<|im_start|>assistant" in chatml:
        assistant_part = chatml.split("<|im_start|>assistant\n")[-1].split("<|im_end|>")[0].strip()
    else:
        assistant_part = row['target_json']
    
    try:
        parsed = json.loads(assistant_part)
        assert "intent" in parsed and "category" in parsed
        valid_count += 1
    except Exception as e:
        raise ValueError(f"JSON validation failed at row {idx}: {e}")

print(f"JSON Schema Validation Passed: 100% ({valid_count}/{len(df)} records valid JSON with intent and category keys).")

### **Task 2.3: Engineer Training and Retrieval Data**

#### **2.3.1 Tokenise Dataset [2 marks]**
**The Task:** Convert text sequences into tensor IDs and apply `-100` masking to padding tokens in the `labels` array.

**Hints & Tips:**
* If a token equals `tokenizer.pad_token_id`, replace it with `-100` in labels.
* `-100` is PyTorch's ignore index — the loss skips these tokens.

**Parameter Tuning:**
* `max_length`: `128` (fast, fits most Bitext prompts), `256` (safer if truncation seen), `64` (too aggressive).

**Learner Inference:** Masking isolates the loss — the model is penalised for wrong JSON, not for padding.

In [ ]:
# YOUR CODE HERE
import torch
from datasets import Dataset

MAX_LENGTH = 128

def tokenize_chatml_function(examples):
    encodings = tokenizer(
        examples['chatml_text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )
    
    labels = []
    for input_id_seq in encodings['input_ids']:
        label_seq = [
            tok if tok != tokenizer.pad_token_id else -100
            for tok in input_id_seq
        ]
        labels.append(label_seq)
    
    encodings['labels'] = labels
    return encodings

hf_dataset = Dataset.from_pandas(df[['instruction', 'intent', 'category', 'chatml_text', 'target_json']])
tokenized_dataset = hf_dataset.map(tokenize_chatml_function, batched=True)
print("Tokenization with -100 label masking completed:")
print(tokenized_dataset)

#### **2.3.2 Analyse Token Distributions [2 marks]**
**The Task:** Analyse the token-count distribution to detect truncation risks and verify the sequence-length configuration.

**Hints & Tips:**
* Count non-padding tokens per row and plot a histogram.
* If >5% of rows hit `max_length`, increase it.
* The JSON target is ~20 tokens — if truncated, the model learns incomplete output.

**Learner Inference:** Sequences hitting the ceiling mean the JSON target is getting cut off, teaching the model to produce broken output.

In [ ]:
# YOUR CODE HERE
import matplotlib.pyplot as plt

token_lengths = [
    len(tokenizer.encode(text, truncation=False))
    for text in df['chatml_text']
]

plt.figure(figsize=(10, 5))
plt.hist(token_lengths, bins=30, color='#3498db', edgecolor='black', alpha=0.8)
plt.axvline(MAX_LENGTH, color='red', linestyle='--', linewidth=2, label=f'max_length={MAX_LENGTH}')
plt.title("ChatML Sequence Token Length Distribution", fontsize=13, fontweight='bold')
plt.xlabel("Sequence Length (Tokens)", fontsize=11)
plt.ylabel("Sample Count", fontsize=11)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("token_distribution.png", dpi=300)
plt.show()

truncated_pct = (sum(1 for l in token_lengths if l > MAX_LENGTH) / len(token_lengths)) * 100
print(f"Max token length in dataset: {max(token_lengths)}")
print(f"Truncation percentage at max_length={MAX_LENGTH}: {truncated_pct:.2f}% (Safe: < 5% threshold).")

#### **2.3.3 Prepare Retrieval Corpus [3 marks]**
**The Task:** Ingest the Markdown SOPs into LangChain Document objects ready to be embedded.

**Hints & Tips:**
* Use `TextLoader` from LangChain.
* Each `.md` file becomes one `Document` with `.page_content` and `.metadata`.
* These documents are embedded into ChromaDB in Notebook 4.

**Learner Inference:** These document objects form the knowledge base your bot retrieves from in Stages 3 and 4.

In [ ]:
# YOUR CODE HERE
from langchain_community.document_loaders import TextLoader
import glob

sop_files = sorted(glob.glob("corporate_policies/*.md") or glob.glob("Dataset/Dataset/sop_documents/*.md"))
sop_documents = []

for fpath in sop_files:
    loader = TextLoader(fpath, encoding='utf-8')
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = os.path.basename(fpath)
        d.metadata["title"] = os.path.basename(fpath).replace(".md", "").replace("_", " ").title()
        sop_documents.extend(docs)

print(f"Prepared LangChain Retrieval Corpus with {len(sop_documents)} documents:")
for doc in sop_documents:
    print(f"  - {doc.metadata['title']} ({len(doc.page_content)} characters)")

### **Task 2.4: Create Dataset Partitions**

#### **2.4.1 Create Train, Validation, and Test Splits [5 marks]**
**The Task:** Split the tokenized dataset into Train (80%), Validation (10%), and Test (10%) partitions using stratified sampling. Demonstrate zero data leakage across all three.

**Hints & Tips:**
* Use `train_test_split` from sklearn with `stratify=df['intent']` so every intent appears in all splits.
* Split twice: first 80/20, then split the 20% into 50/50 (giving 10%/10%).
* The validation set is not optional — it is actively used during fine-tuning (Stage 4 / Notebook 6) for loss monitoring and early stopping.
* The test set must stay completely independent — used only for final evaluation.
* Verify zero overlap with set-intersection checks: `assert len(train_set & test_set) == 0`.
* Build a `df_test` DataFrame preserving readable `text`/`intent`/`category` columns for later scoring.

**Parameter Tuning:**
* Split ratio: `80/10/10` (recommended) or `70/15/15`. Any alternative must be justified.

**Why we are doing it:** To guarantee isolated, unseen data for unbiased evaluation, and a validation set for overfitting detection.
**How we are doing it:** Stratified sklearn splits with explicit programmatic leakage verification.

**Learner Inference:** Testing on data the model trained on gives false confidence (leakage). The held-out test split proves your model handles completely new phrasing; the validation split tells you when to stop training.

In [ ]:
# YOUR CODE HERE
from sklearn.model_selection import train_test_split

# 80/10/10 Stratified Split
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df['intent']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['intent']
)

print(f"Split sizes: Train={len(train_df)} (80%), Val={len(val_df)} (10%), Test={len(test_df)} (10%)")
print(f"Unique intents: Train={train_df['intent'].nunique()}, Val={val_df['intent'].nunique()}, Test={test_df['intent'].nunique()}")

# Zero Data-Leakage Programmatic Set-Intersection Checks
train_prompts = set(train_df['instruction'].str.lower().str.strip())
val_prompts = set(val_df['instruction'].str.lower().str.strip())
test_prompts = set(test_df['instruction'].str.lower().str.strip())

assert len(train_prompts.intersection(val_prompts)) == 0, "Data Leakage between Train and Val!"
assert len(train_prompts.intersection(test_prompts)) == 0, "Data Leakage between Train and Test!"
assert len(val_prompts.intersection(test_prompts)) == 0, "Data Leakage between Val and Test!"

print("Zero Data-Leakage Verification: PASSED (All 3 set intersections are 0).")

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** These cells save all datasets. Notebooks 5, 6, and 7 depend on them.

In [ ]:
# YOUR CODE HERE
ds_train = Dataset.from_pandas(train_df[['instruction', 'intent', 'category', 'chatml_text', 'target_json']])
ds_val = Dataset.from_pandas(val_df[['instruction', 'intent', 'category', 'chatml_text', 'target_json']])
ds_test = Dataset.from_pandas(test_df[['instruction', 'intent', 'category', 'chatml_text', 'target_json']])

tokenized_train = ds_train.map(tokenize_chatml_function, batched=True)
tokenized_valid = ds_val.map(tokenize_chatml_function, batched=True)
tokenized_test = ds_test.map(tokenize_chatml_function, batched=True)

tokenized_train.save_to_disk("./tokenized_train")
tokenized_valid.save_to_disk("./tokenized_valid")
tokenized_test.save_to_disk("./tokenized_test")

test_df.to_csv("df_test.csv", index=False)
print("Saved ./tokenized_train, ./tokenized_valid, ./tokenized_test, and df_test.csv successfully.")

---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify all items before proceeding to Notebook 3.**

- [ ] `sampled_data.csv` loaded
- [ ] **Duplicates + invalid records removed (normalised dedup)** ← _Task 2.1.1_
- [ ] Data standardised (intent=lower, category=UPPER)
- [ ] ChatML formatting applied and validated (valid JSON)
- [ ] Tokenization completed with `-100` label masking
- [ ] Token distribution analysed — truncation risk assessed
- [ ] Retrieval corpus loaded from `.md` files
- [ ] **Stratified Train/Valid/Test split** with all intents in every partition
- [ ] **Zero leakage verified** via set-intersection checks
- [ ] **`./tokenized_train/` + `./tokenized_valid/` saved** ← _CRITICAL for NB6_
- [ ] **`df_test.csv` saved** ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**